In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita renderização gráfica inline no Jupyter notebook
%matplotlib inline

# Como faço o ajuste fino (*fine-tuning*) de um codificador de EEG pré-treinado publicado?

Adapte o checkpoint publicado do CBraMod a pistas reais de olhos abertos/fechados do HBN.
O checkpoint é https://huggingface.co/braindecode/cbramod-pretrained
(documentado pelo modelo CBraMod da Braindecode). Ele foi pré-treinado no TUH EEG;
este exemplo usa a coorte distinta do HBN R5 mini. Baixa cerca de 20 MB
de pesos e dezoito gravações do desafio (aproximadamente 320 MB no total).

Seis partições agrupadas por sujeitos retêm, cada uma, três participantes para teste
enquanto outros três selecionam a época e doze treinam, de modo que cada participante
é avaliado exatamente uma vez. Compare treino do zero (*scratch*), um codificador congelado
com cabeça linear aprendida (*linear probe*) e ajuste fino (*fine-tuning*). O orçamento fixo e reduzido
demonstra a adaptação, não um benchmark de modelo de fundação.


## Antes de começar

Use EEGDash, Braindecode, PyTorch, NumPy, SciPy, scikit-learn e Matplotlib.
O checkpoint público requer acesso à rede no primeiro uso;
``from_pretrained`` armazena seus pesos em cache separadamente de ``EEGDASH_CACHE_DIR``,
que armazena em cache as dezoito gravações de EEG. A revisão abaixo fixa o checkpoint
exato em vez de depender de um branch padrão mutável.

A comparação executável responde a uma pergunta pontual: o que acontece quando a mesma
tarefa posterior é aprendida a partir de pesos aleatórios, de uma representação publicada congelada
ou de uma representação publicada adaptável? Não repete o pré-treinamento em larga escala do checkpoint.
Para a arquitetura e contrato do checkpoint, consulte [o exemplo de modelo pré-treinado da Braindecode](https://braindecode.org/dev/auto_examples/model_building/plot_load_pretrained_models.html).



In [ ]:
# Importa módulos de clonagem profunda, sistema operacional e manipulação de diretórios
import copy
import os
from pathlib import Path

# Importa bibliotecas para plotagem gráfica, tensores e operações numéricas
import matplotlib.pyplot as plt
import numpy as np
import torch
# Importa arquitetura CBraMod e utilitários de pré-processamento e janelamento da Braindecode
from braindecode.models import CBraMod
from braindecode.preprocessing import (
    Preprocessor,
    create_windows_from_events,
    preprocess,
)
# Importa teste de postos sinalizados de Wilcoxon e acurácia balanceada
from scipy.stats import wilcoxon
from sklearn.metrics import balanced_accuracy_score

# Importa classes de dataset do desafio e mapeamento da versão mini
from eegdash import EEGChallengeDataset
from eegdash.const import SUBJECT_MINI_RELEASE_MAP

# Define o diretório de cache persistente
CACHE_DIR = Path(os.environ.get("EEGDASH_CACHE_DIR", "~/.eegdash_cache")).expanduser()
CHANNELS = ["E70", "E75", "E83"]  # três canais posteriores (occipitais)
SFREQ = 200  # CBraMod requer 200 Hz: patches de um segundo com 200 amostras
# Janela de estado estacionário (início, fim) em segundos após cada pista, como no tutorial de olhos abertos/fechados
CUE_WINDOW = {"instructed_toOpenEyes": (5, 19), "instructed_toCloseEyes": (15, 29)}

## Adequar o sinal real à entrada do codificador

E70, E75 e E83 formam um subconjunto explícito e reduzido de canais posteriores, não uma
afirmação de que a montagem completa do HBN seja intercambiável com a montagem de pré-treino.
O derivado do desafio já foi filtrado e amostrado a 100 Hz; reamostrar para 200 Hz satisfaz
o contrato de patches de um segundo e 200 amostras do CBraMod, mas não pode recuperar informações
acima da frequência de Nyquist original.

Duas das vinte gravações da versão mini têm eletrodos posteriores planos ou saturados
(identificados inspecionando amplitudes por canal) e são excluídas por ID. Amplitudes posteriores
ainda diferem por ordens de grandeza entre as gravações restantes, portanto cada gravação é
padronizada por canal com sua própria média e desvio padrão. Essa etapa não usa rótulos e é
aplicada identicamente a cada gravação.

As janelas amostram o estado estacionário de cada condição ocular em vez do transiente
da instrução, seguindo o tutorial de olhos abertos/fechados: sete janelas de dois segundos de
5 a 19 s após uma pista de olhos abertos (o bloco aberto dura 20 s) e de 15 a 29 s após uma pista de
olhos fechados (o bloco fechado dura 40 s). A última pista de olhos abertos ocorre alguns segundos
antes do término da gravação e não consegue fornecer uma janela completa; o Braindecode recusa
essa pista, portanto ela é removida previamente. Rótulos representam as instruções observadas de abrir/fechar,
não rastreamento ocular (*eye tracking*).



In [ ]:
# Carrega gravações de repouso da versão mini HBN R5, excluindo duas com canais saturados/planos
BAD_RECORDINGS = ["NDARAP785CTE", "NDARCA740UC8"]
subjects = [
    s for s in sorted(SUBJECT_MINI_RELEASE_MAP["R5"]) if s not in BAD_RECORDINGS
]
dataset = EEGChallengeDataset(
    release="R5", mini=True, task="RestingState", subject=subjects, cache_dir=CACHE_DIR
)


# Função para remover pistas tardias que não comportam janelas completas
def drop_late_cues(raw):
    """A última pista de olhos abertos ocorre ~5 s antes do fim e não comporta uma janela de 2s."""
    fits = (
        raw.annotations.onset + max(stop for _, stop in CUE_WINDOW.values())
        < raw.times[-1]
    )
    return raw.set_annotations(raw.annotations[fits])


# Aplica pipeline de pré-processamento: canais posteriores, reamostragem a 200 Hz, padronização e limpeza de pistas
preprocess(
    dataset,
    [
        Preprocessor("pick", picks=CHANNELS),
        Preprocessor("resample", sfreq=SFREQ),
        # Padroniza cada canal de cada gravação com sua própria média e desvio padrão (sem uso de rótulos)
        Preprocessor(
            lambda x: (x - x.mean(axis=1, keepdims=True)) / x.std(axis=1, keepdims=True)
        ),
        Preprocessor(drop_late_cues, apply_on_array=False),
    ],
)

# Corta janelas de 2 s: rótulo 0 = olhos abertos, 1 = olhos fechados
windows = create_windows_from_events(
    dataset,
    mapping={"instructed_toOpenEyes": 0, "instructed_toCloseEyes": 1},
    trial_start_offset_samples={
        cue: start * SFREQ for cue, (start, _) in CUE_WINDOW.items()
    },
    trial_stop_offset_samples={
        cue: stop * SFREQ for cue, (_, stop) in CUE_WINDOW.items()
    },
    window_size_samples=2 * SFREQ,
    window_stride_samples=2 * SFREQ,
)

## Reservar participantes, não janelas

``X`` tem formato ``(windows, 3, 400)`` e ``y`` contém as duas classes inteiras
de instrução. Participantes, e não janelas, determinam as partições:
seis partições agrupadas por sujeitos retêm, cada uma, três participantes para teste, outros
três selecionam a época e os doze restantes treinam. Cada participante é testado
exatamente uma vez.

Toda seleção de checkpoint usa a acurácia balanceada de validação. Rótulos de teste são
lidos apenas para a pontuação final de cada regime pré-especificado. Relatar vários
regimes não dá permissão para escolher uma configuração vencedora com base no desempenho de teste
e chamar isso de um novo resultado independente.



In [ ]:
# Monta tensores para PyTorch: X = (janelas, canais, amostras), y = rótulo, subject = participante
X = np.stack([x for x, _, _ in windows]).astype("float32")
metadata = windows.get_metadata()
y = metadata["target"].to_numpy()
subject = metadata["subject"].to_numpy()
# Exibe resumo das dimensões, distribuição de classes e contagem de participantes
print(f"X {X.shape} | classes {np.bincount(y)} | participants {len(subjects)}")

# Cria seis partições de três participantes cada; cada participante será testado exatamente uma vez
folds = np.array_split(np.array(subjects), 6)

## Carregar o checkpoint e compreender as dimensões da cabeça (*head*)

``return_encoder_output=True`` expõe as representações de patch do CBraMod.
Para três canais e dois patches de um segundo, cada um com 200 coordenadas de representação,
o achatamento (*flattening*) fornece ``3 × 2 × 200 = 1200`` entradas para a cabeça linear
de duas classes. A cabeça retorna logits; a entropia cruzada consome logits diretamente,
portanto nenhum softmax é inserido no loop de treinamento.

Se alterar canais ou duração, deduza a nova largura da cabeça a partir de uma passagem
direta (*forward pass*) real do codificador. Apenas alterar ``n_outputs`` não corrige um formato
incompatível de características. O mesmo princípio é ilustrado no
[guia de fine-tuning de modelos de fundação da Braindecode](https://braindecode.org/dev/auto_examples/advanced_training/plot_finetune_foundation_model.html).



In [ ]:
# Faz o download do checkpoint publicado (cerca de 20 MB); a revisão crava os pesos exatos
pretrained = CBraMod.from_pretrained(
    "braindecode/cbramod-pretrained",
    revision="584cdc415913739a05d84bf0c1cb3db397764507",
    return_encoder_output=True,  # retorna representações de patches em vez de pontuações de classe
    n_chans=len(CHANNELS),
    n_times=2 * SFREQ,
    sfreq=SFREQ,
)

## Comparar regimes de otimização com seleção estrita na validação

Três funções pequenas mantêm o loop legível. ``make_model`` constrói o
codificador mais uma cabeça linear: o modelo do zero possui a mesma arquitetura de codificador,
mas sem pesos baixados; o probe linear congela o codificador pré-treinado para que apenas a cabeça
aprenda; e o ajuste fino permite que ambos mudem. ``fit`` treina com AdamW, reembaralha os
minilotes a cada época (caso contrário, as janelas estariam ordenadas por participante e pista,
tornando lotes consecutivos quase de classe única) e mantém os pesos da época com melhor pontuação de
validação. Para o probe, também coloca o codificador congelado em modo de avaliação, pois apenas congelar
gradientes não desativaria seu dropout. ``predict`` retorna a classe prevista de cada janela.

A taxa de aprendizado é ``1e-4`` para os regimes do zero e fine-tune, e ``1e-3`` para a cabeça
linear isolada. Seis épocas por partição mantêm a execução em CPU em poucos minutos; não são um orçamento
de treinamento recomendado. Cada regime é avaliado uma vez por participante retido, como a acurácia
balanceada sobre as janelas daquele participante.



In [ ]:
# Definição dos três regimes comparados
REGIMES = ["scratch", "linear probe", "fine-tune"]


# Função fábrica do modelo: codificador CBraMod + achatamento + camada linear (1200 -> 2)
def make_model(regime):
    """Codificador CBraMod + cabeça linear. 3 canais x 2 patches x 200 características = 1200 entradas."""
    if regime == "scratch":
        encoder = CBraMod(
            n_chans=len(CHANNELS),
            n_times=2 * SFREQ,
            sfreq=SFREQ,
            return_encoder_output=True,
        )
    else:
        encoder = copy.deepcopy(pretrained)
    if regime == "linear probe":
        encoder.requires_grad_(False)  # congela o codificador: apenas a cabeça linear aprende
    return torch.nn.Sequential(encoder, torch.nn.Flatten(), torch.nn.Linear(1200, 2))


# Função de predição em modo de avaliação desativando gradientes
def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.from_numpy(X)).argmax(1).numpy()


# Função de treinamento com AdamW retendo os pesos da melhor época de validação
def fit(model, train, valid, lr, frozen, n_epochs=6, batch_size=16):
    """Treina com AdamW; retorna o modelo com os pesos da época de melhor acurácia na validação."""
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    best_score, best_state = -1, None
    for epoch in range(n_epochs):
        model.train()
        if frozen:
            model[0].eval()  # codificador congelado: também desativa seu dropout
        for batch in torch.randperm(int(train.sum())).split(
            batch_size
        ):  # minilotes aleatorizados
            idx = np.flatnonzero(train)[batch.numpy()]
            loss = torch.nn.functional.cross_entropy(
                model(torch.from_numpy(X[idx])), torch.from_numpy(y[idx])
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        score = balanced_accuracy_score(y[valid], predict(model, X[valid]))
        if score > best_score:
            best_score, best_state = score, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model


# Fixa semente do PyTorch e inicializa dicionário de pontuações por regime
torch.manual_seed(0)
scores = {
    regime: [] for regime in REGIMES
}  # uma acurácia balanceada por participante retido
# Itera pelas 6 partições de validação cruzada externa
for test_subjects in folds:
    others = [s for s in subjects if s not in test_subjects]
    train = np.isin(subject, others[:-3])  # 12 participantes treinam
    valid = np.isin(subject, others[-3:])  # 3 participantes selecionam a melhor época
    test = np.isin(subject, test_subjects)  # 3 participantes retidos avaliados uma única vez
    for regime in REGIMES:
        frozen = regime == "linear probe"
        model = fit(
            make_model(regime), train, valid, lr=1e-3 if frozen else 1e-4, frozen=frozen
        )
        pred = predict(model, X[test])
        for s in test_subjects:
            m = subject[test] == s
            scores[regime].append(balanced_accuracy_score(y[test][m], pred[m]))
    n = len(test_subjects)
    print(
        f"held out {test_subjects.tolist()}: "
        + " | ".join(f"{r} {np.mean(scores[r][-n:]):.2f}" for r in REGIMES)
    )

Estatísticas sobre os participantes (uma pontuação cada) e figura com um ponto por participante



In [ ]:
# Avalia significância estatística de cada regime acima do nível de chance (50%) com teste de Wilcoxon
for regime in REGIMES:
    s = np.array(scores[regime])
    p = wilcoxon(s - 0.5, alternative="greater").pvalue
    print(
        f"{regime:13s} mean {s.mean():.3f} | above chance in {(s > 0.5).sum()}/{len(s)} | Wilcoxon p = {p:.1e}"
    )
# Teste pareado de Wilcoxon comparando fine-tuning vs. treino do zero (scratch)
p = wilcoxon(scores["fine-tune"], scores["scratch"], alternative="greater").pvalue
print(f"fine-tune > scratch (paired over participants): p = {p:.1e}")

# Plota gráfico de barras e dispersão com as pontuações individuais de cada participante
fig, ax = plt.subplots(figsize=(6, 4))
jitter = np.linspace(
    -0.2, 0.2, len(subjects)
)  # espalha os pontos horizontalmente mantendo a ordem dos sujeitos
for i, regime in enumerate(REGIMES):
    ax.bar(i, np.mean(scores[regime]), color="lightgray")
    ax.scatter(i + jitter, scores[regime], s=18, color="k", zorder=3)
ax.axhline(0.5, ls="--", color="gray")  # nível de chance (50%)
ax.set(
    xticks=range(len(REGIMES)),
    xticklabels=REGIMES,
    ylim=(0, 1),
    ylabel="Balanced accuracy per held-out participant",
)
plt.show()

## Ler as barras finais e projetar o próximo experimento

Cada ponto representa um participante retido, avaliado exclusivamente nas janelas daquele
participante; a barra é a média entre os dezoito participantes e a linha tracejada é o nível de chance
para duas classes. Pesos aleatórios permanecem próximos ao acaso. A representação publicada congelada
já separa os dois estados oculares para a maioria dos participantes, e o ajuste fino obtém o melhor resultado.
O teste pareado entre ajuste fino e treino do zero é a evidência de que os pesos pré-treinados importam,
pois os dois regimes compartilham partições, janelas, orçamento e semente. Alguns participantes situam-se
próximos de 0.5 em todos os regimes; essas gravações mostram pouca reatividade de alfa nos canais selecionados,
o que é uma propriedade dos dados a inspecionar, não uma falha de modelagem a ser contornada por ajustes.

Para uma comparação mais robusta, adicione sementes repetidas e pré-especifique orçamentos mais longos.
Compare inicializações idênticas de cabeça ao estimar o efeito dos pesos pré-treinados. Inspecione
matrizes de confusão por participante e qualidade dos canais antes de atribuir uma mudança ao codificador.
Salve os pesos selecionados juntamente com a revisão do checkpoint, ordem de canais, taxa, deslocamentos de janelas,
regra de padronização e listas de sujeitos caso estenda esta página para um modelo treinado reutilizável;
pesos isolados omitem o contrato de entrada.

